In [1]:
import numpy as np
from sklearn.externals.array_api_extra.testing import override

from model_wrapper import *
import cuml.accel
from math import floor
cuml.accel.install()

# of Training Instances: 47
# of Testing Instances: 11
Current RAM usage: 299.18 MB


In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.decomposition import IncrementalPCA

class LogRegModel(Model):
    def __init__(self, anatomical_plane, fluid_sensitive=None, fat_suppression=None, n_comp=20):
        self.n_comp = n_comp
        self.inc_pca = IncrementalPCA(n_components=n_comp)
        self.model = OneVsRestClassifier(LogisticRegression(max_iter=1500))
        super().__init__(anatomical_plane, fluid_sensitive, fat_suppression)

    @override
    def full_fit(self, x: np.ndarray, y: np.ndarray):
        x = np.reshape(x, shape=(x.shape[0], x.shape[1] * x.shape[2] * x.shape[3]))
        n_batches = floor(x.shape[0] / self.n_comp)

        for X_batch in np.array_split(x, n_batches):
            self.inc_pca.partial_fit(X_batch)

        x_reduced = self.inc_pca.transform(x)
        self.model.fit(x_reduced, y)

    @override
    def predict_batch(self, x: np.ndarray) -> np.ndarray:
        x = np.reshape(x, shape=(x.shape[0], x.shape[1] * x.shape[2] * x.shape[3]))
        x_reduced = self.inc_pca.transform(x)
        return self.model.predict(x_reduced)

    @override
    def predict_instance(self, x: np.ndarray) -> np.ndarray:
        x = np.reshape(x, shape=(1, x.shape[0] * x.shape[1] * x.shape[2]))
        x_reduced = self.inc_pca.transform(x)
        pred_ = self.model.predict(x_reduced)
        return np.reshape(pred_, shape=(pred_.shape[1]))


In [3]:
# Depth 1 ensemble
ensemble = [LogRegModel(p) for p in planes]
scores = Model.get_ensemble_auc_score(ensemble, 1)

print(f"Model Score: {np.mean(scores)}")
for i in range(len(target_columns)):
    print(f"\t{target_columns[i]}: {scores[i]}")

Training Model: 
	Plane: Sagittal
	Fluid Sensitive: None
	Fat Suppression: None
	Training Shape: (64, 12, 512, 512)
	Validation Shape: (43, 12, 512, 512)
	AUC Score: 0.6244025168159355
		ACL: 0.6118421052631579
		MCL: 0.7094594594594594
		Medial Meniscus: 0.6011111111111112
		Lateral Meniscus: 0.6369047619047619
		Medial OA: 0.6196969696969697
		Lateral OA: 0.5575396825396824
		PF OA: 0.6626344086021506
		Effusion: 0.5380434782608696
		Synovitis: 0.5231481481481481
		Baker's: 0.7464285714285714
		Contusion: 0.5333333333333333
		Fracture: 0.7526881720430106

Training Model: 
	Plane: Axial
	Fluid Sensitive: None
	Fat Suppression: None
	Training Shape: (44, 12, 512, 512)
	Validation Shape: (22, 12, 512, 512)
	AUC Score: 0.5668559195648979
		ACL: 0.4572649572649573
		MCL: 0.4473684210526316
		Medial Meniscus: 0.40909090909090906
		Lateral Meniscus: 0.5897435897435898
		Medial OA: 0.5476190476190477
		Lateral OA: 0.8117647058823529
		PF OA: 0.6
		Effusion: 0.4910714285714286
		Synovitis: 0.

In [4]:
print_memory_usage()

Current RAM usage: 4709.94 MB


In [5]:
# Depth 2 ensemble
# fluid sensitive and fat suppression can either be 0 or 1
ensemble = []

for p in planes:
    for i in range(2):
        ensemble.append(LogRegModel(p, i, i, n_comp=8))

scores = Model.get_ensemble_auc_score(ensemble, 2)

print(f"Model Score: {np.mean(scores)}")
for i in range(len(target_columns)):
    print(f"\t{target_columns[i]}: {scores[i]}")

Training Model: 
	Plane: Sagittal
	Fluid Sensitive: 0
	Fat Suppression: 0
	Training Shape: (39, 12, 512, 512)
	Validation Shape: (18, 12, 512, 512)
	AUC Score: 0.4817911255411255
		ACL: 0.41666666666666663
		MCL: 0.4
		Medial Meniscus: 0.425
		Lateral Meniscus: 0.35
		Medial OA: 0.5
		Lateral OA: 0.5
		PF OA: 0.5
		Effusion: 0.37500000000000006
		Synovitis: 0.475
		Baker's: 0.5
		Contusion: 0.5064935064935064
		Fracture: 0.8333333333333333

Training Model: 
	Plane: Sagittal
	Fluid Sensitive: 1
	Fat Suppression: 1
	Training Shape: (33, 12, 512, 512)
	Validation Shape: (17, 12, 512, 512)
	AUC Score: 0.47918620731120737
		ACL: 0.4513888888888889
		MCL: 0.5
		Medial Meniscus: 0.5763888888888888
		Lateral Meniscus: 0.5142857142857142
		Medial OA: 0.5
		Lateral OA: 0.5
		PF OA: 0.36363636363636365
		Effusion: 0.43939393939393945
		Synovitis: 0.39583333333333337
		Baker's: 0.5
		Contusion: 0.4714285714285714
		Fracture: 0.5378787878787878

Training Model: 
	Plane: Axial
	Fluid Sensitive: 0
	F

In [6]:
print_memory_usage()

Current RAM usage: 4974.84 MB
